In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load the Credit Data

In [ ]:
filename = 'default of credit card clients.csv'
credit_data = pd.read_csv(filename)

In [ ]:
credit_data.head()

### Prepare the Dataset

In [ ]:
credit_data.drop(['ID'], axis=1, inplace=True)

In [ ]:
# Identify number of categories in all categorical variables
categorical_list = ['SEX', 'EDUCATION', 'MARRIAGE']
credit_data[categorical_list].nunique()

In [ ]:
# One-hot encode
credit_cat_cols = pd.get_dummies(credit_data, columns = categorical_list)

In [ ]:
credit_cat_cols.head()

In [ ]:
credit_cat_cols.columns

In [ ]:
y = credit_cat_cols['default payment next month'].to_numpy()
X = credit_cat_cols.drop(['default payment next month'], axis=1)

### Test-Train-Validation Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)
X_test, X_valid, y_test, y_valid = train_test_split(X_test, y_test, test_size=0.5, random_state=36)

### Standard Scaling

In [ ]:
standard_scaler = StandardScaler()
scaler_cols = list(set(credit_data.columns) - set(categorical_list))
scaler_cols.remove('default payment next month')
X_train_scaled = standard_scaler.fit_transform(X_train[scaler_cols])
X_train[scaler_cols] = X_train_scaled

In [ ]:
X_test_scaled = standard_scaler.transform(X_test[scaler_cols])
X_test[scaler_cols] = X_test_scaled

In [ ]:
X_valid_scaled = standard_scaler.transform(X_valid[scaler_cols])
X_valid[scaler_cols] = X_valid_scaled

In [ ]:
input_cols = X_train.shape[1]
input_cols

In [ ]:
no_epochs = 20
batch_size = 1024

In [ ]:
train_x = torch.Tensor(X_train.to_numpy().astype(np.float32).copy()).to(device)
train_y = torch.Tensor(y_train).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, 
                                           shuffle=True, drop_last=True)

val_x = torch.Tensor(X_valid.to_numpy().astype(np.float32).copy()).to(device)
val_y = torch.Tensor(y_valid).to(device)
val_dataset = TensorDataset(val_x, val_y)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, 
                                          shuffle=False, drop_last=True)

### Build Model

In [ ]:
def l1_regularizer(model):
    lossl1 = 0
    for model_param_name, model_param_value in model.named_parameters():
            if model_param_name.endswith('weight'):
                lossl1 +=  torch.norm(model_param_value, 1)
    return lossl1   

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs, 
                level=0.5, reg_type=None, reg_lambda=0.01, early_stopping_patience=None):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []
    best_test_loss = float('inf')
    epochs_no_improve = 0

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y.float())

            # Regularization
            if reg_type == 'L1':
                l1_norm = l1_regularizer(model)
                loss = loss + reg_lambda * l1_norm

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)
            predicted = outputs.squeeze() > level
            correct_train += (predicted == batch_y).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)

        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y.float())
                test_loss += loss.item() * batch_X.size(0)
                predicted = outputs.squeeze() > level
                correct_test += (predicted == batch_y).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, \
                                   Test Loss: {test_loss:.4f}, Train Acc: {train_accuracy:.2f}%, \
                                   Test Acc: {test_accuracy:.2f}%")

        # Early Stopping
        if early_stopping_patience is not None:
            if test_loss < best_test_loss:
                best_test_loss = test_loss
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve == early_stopping_patience:
                    print(f"Early stopping triggered after {epoch + 1} epochs.")
                    break

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies

    return history


In [ ]:
class SmallBinaryClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_activation='sigmoid'):
        super(SmallBinaryClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, 1)

        if output_activation == 'sigmoid':
            self.output_activation = nn.Sigmoid()
        elif output_activation == 'tanh':
            self.output_activation = nn.Tanh()
        else:
            raise ValueError("Invalid output activation. Choose 'sigmoid' or 'tanh'.")

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.output_layer(x)
        x = self.output_activation(x)
        return x


In [ ]:
history_dict = dict()

### Original Small Model

In [ ]:
bce_model = SmallBinaryClassifier(input_cols, 50).to(device)

In [ ]:
bce_loss_fn = nn.BCELoss()
bce_optimizer = optim.Adam(bce_model.parameters(), lr=0.001)

In [ ]:
history_dict['Small Model'] = train_model(bce_model, train_loader, val_loader,
                                          bce_loss_fn, bce_optimizer, no_epochs)

In [ ]:
class BinaryClassifier(nn.Module):
    def __init__(self, input_size, dropout_type=None, dropout_rate=0.5):
        super(BinaryClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, 100)
        self.fc2 = nn.Linear(100, 100)
        self.fc3 = nn.Linear(100, 100)
        self.fc4 = nn.Linear(100, 100)
        self.fc5 = nn.Linear(100, 100)
        self.output_layer = nn.Linear(100, 1)
        self.relu = nn.ReLU()
        self.dropout_type = dropout_type
        self.dropout_rate = dropout_rate
        self.output_activation = nn.Sigmoid()


    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self._apply_dropout(x)

        x = self.fc2(x)
        x = self.relu(x)
        x = self._apply_dropout(x)

        x = self.fc3(x)
        x = self.relu(x)
        x = self._apply_dropout(x)

        x = self.fc4(x)
        x = self.relu(x)
        x = self._apply_dropout(x)

        x = self.fc5(x)
        x = self.relu(x)
        x = self._apply_dropout(x)

        x = self.output_layer(x)
        x = self.output_activation(x)
        return x

    def _apply_dropout(self, x):
        if self.dropout_type == 'standard':
            return F.dropout(x, self.dropout_rate, training=self.training)
        elif self.dropout_type == 'gaussian':
            return F.dropout2d(x, self.dropout_rate, training=self.training) if self.training else x
        return x


### No Regularization

In [ ]:
model = BinaryClassifier(input_cols).to(device)

In [ ]:
loss_fn = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
history_dict['None'] = train_model(model, train_loader, val_loader,
                                 loss_fn, optimizer, no_epochs)

### Weight Decay L2

In [ ]:
l2_model = BinaryClassifier(input_cols).to(device)

In [ ]:
l2_loss_fn = nn.BCELoss()
l2_optimizer = optim.Adam(l2_model.parameters(), lr=0.001, weight_decay=0.01)

In [ ]:
history_dict['L2'] = train_model(l2_model, train_loader, val_loader,
                                 l2_loss_fn, l2_optimizer, no_epochs)

### Weight Decay L1

In [ ]:
l1_model = BinaryClassifier(input_cols).to(device)

In [ ]:
l1_loss_fn = nn.BCELoss()
l1_optimizer = optim.Adam(l1_model.parameters(), lr=0.001)

In [ ]:
history_dict['L1'] = train_model(l1_model, train_loader, val_loader,
                                 l1_loss_fn, l1_optimizer, no_epochs, reg_type='L1',reg_lambda=0.001)

### Dropout

In [ ]:
drop_model = BinaryClassifier(input_cols, 'standard').to(device)

In [ ]:
drop_loss_fn = nn.BCELoss()
drop_optimizer = optim.Adam(drop_model.parameters(), lr=0.001)

In [ ]:
history_dict['Dropout'] = train_model(drop_model, train_loader, val_loader,
                                 drop_loss_fn, drop_optimizer, no_epochs)

### Gaussian Dropout

In [ ]:
gdrop_model = BinaryClassifier(input_cols, 'gaussian').to(device)

In [ ]:
gdrop_loss_fn = nn.BCELoss()
gdrop_optimizer = optim.Adam(gdrop_model.parameters(), lr=0.001)

In [ ]:
history_dict['Gaussian Dropout'] = train_model(gdrop_model, train_loader, val_loader,
                                               gdrop_loss_fn, gdrop_optimizer, no_epochs)

### Early Stopping

In [ ]:
es_model = BinaryClassifier(input_cols).to(device)

In [ ]:
es_loss_fn = nn.BCELoss()
es_optimizer = optim.Adam(es_model.parameters(), lr=0.001)

In [ ]:
history_dict['Early Stopping'] = train_model(es_model, train_loader, val_loader,
                                             es_loss_fn, es_optimizer, no_epochs, early_stopping_patience=5)

### Plot Validation Accuracy

In [ ]:
epochs = range(1, no_epochs + 1)

In [ ]:
def smooth_curve(points, factor=0.8):
    smoothed_points = []
    for point in points:
        if smoothed_points:
            previous = smoothed_points[-1]
            smoothed_points.append(previous * factor + point * ( 1 - factor))
        else:
            smoothed_points.append(point)
    return smoothed_points

In [ ]:
fig, ax = plt.subplots(figsize=(14,10))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']) * cycler('lw', [1, 2]))

ax.set_prop_cycle(monochrome)
for ilr in history_dict:
    valacc_hist = history_dict[ilr]
    val_acc_values = valacc_hist['test_acc']
    epochs = range(1, len(val_acc_values) + 1)
    ax.plot(epochs, smooth_curve(val_acc_values), label = str(ilr))

ax.set_xlabel('Epochs')
ax.set_ylabel('Validation Accuracy')
ax.legend()
fig.savefig('TestTaiwaneseReg2.png', dpi=300, bbox_inches='tight')